# NF1

Cohort: ``NF1``, Phenopacket Store v0.1.27 (n = 405).

All 405 individuals are annotated with the same disease:

- Neurofibromatosis, type 1, ``OMIM:162200``, n = 405

The NF1 cohort therefore corresponds to a single disease group for the
disease-level phenotype correlation analysis.

In [1]:
from contextlib import contextmanager
from pathlib import Path
from zipfile import ZipFile
import os
import pandas as pd
import math

from ppktstore.model import PhenopacketStore

from phenocor import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
)

PHENOPACKET_STORE_VERSION = "0.1.27"
HPO_RELEASE = "v2026-06-23"
FDR_THRESHOLD = 0.05
ASSOCIATION_MIN_EFFECTIVE_N = 20

# Set PHENOSIGN_PROJECT_ROOT or edit this default if the repository is elsewhere.
PROJECT_ROOT = Path(
    os.environ.get("PHENOSIGN_PROJECT_ROOT", "..")
).expanduser().resolve()
STORE_ZIP = PROJECT_ROOT / "data" / "all_phenopackets.zip"

COHORT = "NF1"
CASE_DIR = PROJECT_ROOT / "results" / COHORT

if not STORE_ZIP.is_file():
    raise FileNotFoundError(f"Phenopacket Store ZIP not found: {STORE_ZIP}")

@contextmanager
def open_local_phenopacket_store(zip_path):
    with ZipFile(zip_path, mode="r") as zip_file:
        store = PhenopacketStore.from_release_zip(
            zip_file=zip_file,
            strategy="lazy",
        )
        yield store

In [2]:
with open_local_phenopacket_store(STORE_ZIP) as store:
    phenopackets = list(store.iter_cohort_phenopackets(COHORT))

print(f"Loaded {len(phenopackets)} phenopackets")

dataset = PhenotypeDatasetBuilder(
    phenopackets,
    hpo_release=HPO_RELEASE,
).build()

disease_ids = (
    dataset.get_diseases()
    .explode()
    .dropna()
    .unique()
)

print(f"Number of diseases: {len(disease_ids)}")
print(disease_ids)

Loaded 405 phenopackets
Number of diseases: 1
['OMIM:162200']


## Pairwise correlation

In [4]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.correlation_results
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

Calculating pairwise correlations: 100%|██████████| 477/477 [00:00<00:00, 2898.89it/s]


tested pairs: 412 | significant: 27


,HPO_A,HPO_B,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,HPO_A_label,HPO_B_label,n_pmids,pmids
0,HP:0000997,HP:0030052,0.674945,5.245645e-28,2.161206e-25,109,6,35,86,236,Axillary freckling,Inguinal freckling,8,17160901;20513137;26178382;29290338;36659944;3...
1,HP:0001067,HP:0009737,0.496283,6.359971e-16,1.310154e-13,146,21,42,62,271,Neurofibroma,Lisch nodules,8,17160901;20513137;26178382;29290338;36659944;3...
2,HP:0009732,HP:0009735,0.725802,2.920424e-15,4.010716e-13,254,4,5,13,276,Plexiform neurofibroma,Spinal neurofibroma,5,17160901;20513137;26178382;29290338;38596211
3,HP:0000997,HP:0001067,0.391838,1.283969e-09,1.322488e-07,87,27,46,77,237,Axillary freckling,Neurofibroma,9,17160901;20513137;26178382;29290338;36659944;3...
4,HP:0001067,HP:0030052,0.363914,4.097005e-08,2.813277e-06,101,32,40,60,233,Neurofibroma,Inguinal freckling,8,17160901;20513137;26178382;29290338;36659944;3...


In [6]:
results.plot_correlation_heatmap_with_significance(
    title="Cohort NF1",
)

In [7]:
SUPERSCRIPT_TRANSLATION = str.maketrans(
    "0123456789-",
    "⁰¹²³⁴⁵⁶⁷⁸⁹⁻",
)

def format_p_value(value: float) -> str:
    """Format p values using Unicode superscripts for manuscript tables."""
    if value >= 0.001:
        return f"{value:.3f}"

    exponent = math.floor(math.log10(value))
    coefficient = value / (10 ** exponent)
    exponent_text = str(exponent).translate(SUPERSCRIPT_TRANSLATION)

    return f"{coefficient:.2f}×10{exponent_text}"


def phenotype_name(row: pd.Series, term: str) -> str:
    """Combine an HPO label and identifier into one table field."""
    return f"{row[f'HPO_{term}_label']} ({row[f'HPO_{term}']})"


def format_p_values(row: pd.Series) -> str:
    """
    First line: BH FDR-adjusted P value.
    Second line in parentheses: Fisher's exact P value.
    """
    return (
        f"{format_p_value(row['adj_p_value'])}\n"
        f"({format_p_value(row['p_value'])})"
    )


# All FDR-significant phenotype-state associations.
all_significant_associations = (
    t.loc[t["adj_p_value"] < FDR_THRESHOLD]
    .copy()
    .sort_values(
        by=["adj_p_value", "correlation"],
        ascending=[True, False],
    )
)


def prepare_association_table(
    association_df: pd.DataFrame,
    include_publication_count: bool,
) -> pd.DataFrame:
    """Create a manuscript-facing association table."""
    table = pd.DataFrame(
        {
            "Phenotype A": association_df.apply(
                phenotype_name,
                axis=1,
                term="A",
            ),
            "Phenotype B": association_df.apply(
                phenotype_name,
                axis=1,
                term="B",
            ),
            "Phi": association_df["correlation"].map(
                lambda value: f"{value:.2f}"
            ),
            "Total": association_df["n_individuals"].astype(int),
            "State counts (O/O; O/E; E/O; E/E)": (
                association_df["n(A:O/B:O)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:O/B:E)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:E/B:O)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:E/B:E)"].astype(int).astype(str)
            ),
            "FDR-adjusted P (Fisher's exact P)": association_df.apply(
                format_p_values,
                axis=1,
            ),
        }
    )

    if include_publication_count:
        table["PMIDs (n)"] = association_df[
            "n_pmids"
        ].astype(int)

    return table.reset_index(drop=True)


# Main-text Table 2: selected high-magnitude associations.
main_table_2 = prepare_association_table(
    all_significant_associations,
    include_publication_count=True,
)


# Save publication-ready tables without pandas row indices.
main_table_2.to_csv(
    CASE_DIR / "correlation_results_NF1.csv",
    index=False,
)


print(
    f"FDR-significant pairs: {len(all_significant_associations)}\n"
)

display(main_table_2)

FDR-significant pairs: 27



,Phenotype A,Phenotype B,Phi,Total,State counts (O/O; O/E; E/O; E/E),FDR-adjusted P (Fisher's exact P),PMIDs (n)
0,Axillary freckling (HP:0000997),Inguinal freckling (HP:0030052),0.67,236,86; 35; 6; 109,2.16×10⁻²⁵\n(5.25×10⁻²⁸),8
1,Neurofibroma (HP:0001067),Lisch nodules (HP:0009737),0.50,271,62; 42; 21; 146,1.31×10⁻¹³\n(6.36×10⁻¹⁶),8
2,Plexiform neurofibroma (HP:0009732),Spinal neurofibroma (HP:0009735),0.73,276,13; 5; 4; 254,4.01×10⁻¹³\n(2.92×10⁻¹⁵),5
3,Axillary freckling (HP:0000997),Neurofibroma (HP:0001067),0.39,237,77; 46; 27; 87,1.32×10⁻⁷\n(1.28×10⁻⁹),9
4,Neurofibroma (HP:0001067),Inguinal freckling (HP:0030052),0.36,233,60; 40; 32; 101,2.81×10⁻⁶\n(4.10×10⁻⁸),8
5,Neurofibroma (HP:0001067),Scoliosis (HP:0002650),0.30,351,39; 91; 16; 205,2.81×10⁻⁶\n(3.48×10⁻⁸),9
6,Freckling (HP:0001480),Multiple cafe-au-lait spots (HP:0007565),0.30,317,205; 14; 70; 28,2.00×10⁻⁵\n(3.39×10⁻⁷),8
7,Pectus carinatum (HP:0000768),Scoliosis (HP:0002650),0.42,252,7; 2; 20; 223,1.21×10⁻⁴\n(2.34×10⁻⁶),3
8,Axillary freckling (HP:0000997),Multiple cafe-au-lait spots (HP:0007565),0.30,234,118; 7; 79; 30,1.88×10⁻⁴\n(4.10×10⁻⁶),8
9,Neurofibroma (HP:0001067),Freckling (HP:0001480),0.22,375,126; 23; 146; 80,7.68×10⁻⁴\n(1.87×10⁻⁵),9
